[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/triality/triality.ipynb)

# Spin(8) Triality

This example computes Spin(8) triality, the symmetry that links three different rotations of eight-dimensional space through octonion multiplication: rotating a product of two octonions agrees with multiplying the factors rotated by two other rotations, even where none of the three preserves multiplication on its own. The rotations for which all three agree are the symmetries of the octonions themselves, the exceptional group G2.

An octonion is modelled as a scalar plus a vector of seven anticommuting units, and the product as a map of two octonions, `multiply`, built from the seven oriented lines of the Fano plane. Every infinitesimal rotation of the eight components splits into a left multiplication, a right multiplication and a derivation, which obeys the product rule. Minus the trace of a composition pairs skew maps, so one solve against the two multiplication families finds the split, and with it three actions for each of the 28 generators that satisfy `output(multiply) == multiply(left, Octonion) + multiply(Octonion, right)`. Where both multiplication parts vanish, 14 generators remain: those of G2.

In the notation of Lie theory the three actions are the vector and the two half-spin representations of $\mathfrak{so}(8)$, exchanged by the outer automorphisms that permute the outer nodes of the $D_4$ diagram, and the product rule reads as $\theta_1(xy) = \theta_2(x)\,y + x\,\theta_3(y)$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.math.triality import render

np.set_printoptions(precision=4, suppress=True)

In [ ]:
ga = Algebra("a-b-c-d-e-f-g-")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Trivector = ga.gatype.trivector()
Octonion = ga.gatype.from_grades([0, 1])
Rate = ga.gatype.from_grades([1, 2])

## 1. Octonion multiplication

An octonion occupies the scalar and vector parts of the algebra. Seven oriented triples specify its multiplication. The squared length is `value.involute().scalar_product(value)`: involution negates the vector part, compensating for its negative square. Octonion multiplication preserves this positive length, but its brackets matter.

In [ ]:
# Each triple supplies one quaternion multiplication cycle.
cycles = stack([mv.abc, mv.ade, mv.agf, mv.bdf, mv.beg, mv.cdg, mv.cfe])        # [triples] Trivector
# The seven oriented lines of the Fano plane.
fano = cycles.sum(axis=0)                                                    # Trivector
# The projection is part of the octonion product.
multiply = ((Octonion * Octonion) * (1 - fano)).cast(Octonion)          # Octonion <- (Octonion, Octonion)

## 2. Three linked rotations

The output rotation turns the plane spanned by the scalar and `mv.a`, fixing the other six directions. Left and right multiplication by the same half-angle unit give its two companions, each turning four planes through half the angle. Their different actions agree through the product: `output_rotation(multiply)` equals `multiply(left_rotation, right_rotation)`. This is the octonion Moufang identity, an equality of two maps of two octonions.

In [ ]:
angle = np.pi / 2
direction = mv.a                                                             # Vector
first = 1 + mv.b                                                            # Octonion
second = mv.c + mv.d                                                        # Octonion

In [ ]:
unit = (direction * (angle / 2)).exp()                                       # Octonion
left_rotation = multiply(unit, Octonion)                                     # Octonion <- Octonion
right_rotation = multiply(Octonion, unit)                                    # Octonion <- Octonion
output_rotation = multiply(unit, multiply(Octonion, unit))                   # Octonion <- Octonion

for name, action in (("Output rotation", output_rotation),
                     ("First-factor rotation", left_rotation),
                     ("Second-factor rotation", right_rotation)):
    print(name)
    print(action.coefficient_table())

rotated_product = output_rotation(multiply(first, second))                   # Octonion
product_of_rotated = multiply(left_rotation(first), right_rotation(second))  # Octonion
print("Rotate the product")
print(rotated_product.coefficient_table())
print("Multiply the rotated factors")
print(product_of_rotated.coefficient_table())

## 3. One turn and two turns

After one full turn the output action is the identity, while both factor actions are minus the identity. Two full turns return all three. The opposite signs on the two factors cancel in their product. Taking the output as the vector representation of Spin(8), the companions are its two chiral spinor representations; each has eight real components.

In [ ]:
turn_angles = np.array([0, 2 * np.pi, 4 * np.pi])
turn_names = ("Start", "One turn", "Two turns")
action_names = ("Output", "First factor", "Second factor")

In [ ]:
units = (direction * (turn_angles / 2)).exp()                                # [turns] Octonion
left_returns = multiply(units, Octonion)                                     # [turns] Octonion <- Octonion
right_returns = multiply(Octonion, units)                                    # [turns] Octonion <- Octonion
output_returns = multiply(units, multiply(Octonion, units))                   # [turns] Octonion <- Octonion

# At these angles each map is a scalar multiple of the identity.
# Dividing its trace by the dimension reads that multiple.
identity_trace = Octonion.trace()
return_factors = stack([output_returns.trace(), left_returns.trace(),
                        right_returns.trace()]) / identity_trace            # [actions, turns] Scalar
print(render.return_table(return_factors, action_names, turn_names))

## 4. Recovering the general actions

Every infinitesimal rotation of the eight components splits into left multiplication, right multiplication and a common part obeying the product rule. The common part is called a derivation. Minus the trace of a composition measures overlap between skew maps, so solving the two multiplication families' trace form finds the pieces to remove.

In [ ]:
# Twenty-one turns among the vector components and seven involving the scalar.
turn = (Rate.restrict.bivector().commutator(Octonion)
        + Rate.restrict.vector().anticommutator(Octonion))                   # Octonion <- (Rate, Octonion)

left_multiply = multiply(Vector, Octonion)                                   # Octonion <- (Vector, Octonion)
# Conjugation reverses a product; both families take the multiplying vector first.
right_multiply = -multiply(Vector, Octonion.involute()).involute()            # Octonion <- (Vector, Octonion)
multiplications = stack([left_multiply, right_multiply])                     # [sides] Octonion <- (Vector, Octonion)

# The two sites distinguish left and right multiplication.
projection_form = (
    -multiplications[:, None](Vector, multiplications[None, :]).trace(0, 3)
).field(1, 2)                                                               # Scalar <- (Vector[2], Vector[2])
overlap = (-turn(Rate, multiplications).trace(0, 3)).field(2)                 # Scalar <- (Rate, Vector[2])
offsets = projection_form.solve(overlap)                                    # Vector[2] <- Rate

read_left = (mv.scalar([[1], [0]]) * Vector).field(1)                         # Vector <- Vector[2]
read_right = (mv.scalar([[0], [1]]) * Vector).field(1)                        # Vector <- Vector[2]

## 5. The common G2 action

The two multiplication offsets measure how far a rotation departs from a derivation. Their squared lengths give a nonnegative form on rates. Its zero modes recover the common actions, while all its eigendirections supply a complete frame of rotation generators.

In [ ]:
tolerance = 1e-10

In [ ]:
constraint = offsets.involute().scalar_product(offsets).sites.sum()        # Scalar <- (Rate, Rate)
# Clifford conjugation gives a positive pairing on both grades of Rate.
metric = Rate.clifford_conjugate().scalar_product(Rate)                      # Scalar <- (Rate, Rate)
values, rates = constraint.eigh(metric)                                     # [generators] Scalar, Rate
common_rates = rates[values < tolerance]                                    # [common generators] Rate

print(f"Triality generators: {len(rates)}")
print(f"Common generators: {len(common_rates)}")

## 6. The product rule for all generators

The derivation acts equally in all three places. Redistributing the two multiplication pieces supplies the companions for every rotation generator. Their combined action obeys the product rule; exponentiating the three actions together therefore preserves the finite triality identity.

In [ ]:
output = turn(rates, Octonion)                                              # [generators] Octonion <- Octonion
left_offset = read_left(offsets(rates))                                      # [generators] Vector
right_offset = read_right(offsets(rates))                                    # [generators] Vector
derivation = (output - multiply(left_offset, Octonion)
              - multiply(Octonion, right_offset))                           # [generators] Octonion <- Octonion

left = (derivation + multiply(left_offset, Octonion)
        + multiply(Octonion, left_offset - right_offset))                    # [generators] Octonion <- Octonion
right = (derivation + multiply(right_offset - left_offset, Octonion)
         + multiply(Octonion, right_offset))                                 # [generators] Octonion <- Octonion

product_rule = (output(multiply) - multiply(left, Octonion)
                - multiply(Octonion, right))                                # [generators] Octonion <- (Octonion, Octonion)

print("Squared product-rule residual over all generators and arguments")
print(product_rule.involute().scalar_product(product_rule(Octonion.involute(), Octonion.involute()))
      .contract(1, 3).contract(1, 2).sum(axis=0).coefficient_table())

In [ ]:
# checks
# Every generator's three actions obey the product rule, and fourteen generators are common to all three.
np.testing.assert_allclose(product_rule.kernel, 0, atol=1e-11)
assert len(rates) == 28 and len(common_rates) == 14

The 28 linked generators form the Lie algebra of Spin(8); the 14 common generators form that of G2. Triality allows the vector and the two spinor representations to exchange roles through automorphisms of Spin(8). It is the continuous symmetry behind the three interchangeable outer nodes of the D4 diagram. John Baez develops the [triality construction](https://math.ucr.edu/home/baez/octonions/node7.html) and the [decomposition into G2 and two multiplication families](https://math.ucr.edu/home/baez/octonions/node14.html) in *The Octonions*.